# Irish Home Prices — Optuna + MLflow — Option A & B
### A Beginner-Friendly Guide to ML Pipelines, Hyperparameter Tuning, and Model Registration

---

## 🗺️ What This Notebook Teaches You

This notebook walks through a **complete, real-world machine learning workflow** using Irish property price data. By the end you will understand:

1. **How to build an ML pipeline** — a chain of steps that transforms raw data and makes predictions
2. **What Optuna does** — automatically finds the best settings (hyperparameters) for your model
3. **What MLflow does** — tracks experiments and stores models so you can retrieve them later
4. **Option A vs Option B** — two different ways to package your preprocessing and model for production

---

## 🧠 Key Concepts — Read This First!

### What is a Machine Learning Pipeline?
Think of it like an assembly line in a factory. Raw materials (your data) go in one end, and a finished product (a prediction) comes out the other. Each station on the line does one job: cleaning missing values, converting text to numbers, scaling numbers to a common range, then finally making a prediction. Scikit-learn's `Pipeline` object lets you chain all these steps together so they always run in the right order.

### What is Hyperparameter Tuning?
A LightGBM model has dozens of dials you can turn — how many trees to build, how deep each tree should be, how fast it should learn. The right combination of dial settings makes the model accurate; the wrong combination makes it poor. **Optuna** is an automated search tool that tries many combinations intelligently and tells you which one worked best.

### What is MLflow?
Imagine running 50 experiments but forgetting which settings produced your best model. MLflow is a logbook. Every time you train a model, MLflow records: the settings you used, the accuracy you got, and the model itself. Later, you can load any model from this logbook with a single line of code — even months later, on a different machine.

### Option A vs Option B — The Core Question
When you send new data to your deployed model for predictions, the data must be preprocessed (cleaned, scaled, encoded) exactly the same way it was during training. There are two ways to guarantee this:

- **Option A:** Bundle the preprocessing steps *inside* the model using scikit-learn's `Pipeline`. They are inseparable — the model cannot run without its preprocessing.
- **Option B:** Save the preprocessing as a *separate file*, then write a custom `pyfunc` wrapper that loads both files and runs them in sequence. This is more flexible — you can swap out the preprocessor independently of the model.

---

## 📊 The Two Prediction Tasks

| Task | Type | What We Predict | Algorithm |
|------|------|-----------------|-----------|
| **Regression** | Continuous number | Actual price in euros | LightGBM |
| **Classification** | Category | Is this property "expensive"? (top 25%) | LightGBM |

---

## ⚙️ Speed Controls (Lightweight Knobs)

These three settings control how fast the notebook runs. They are deliberately set small so you can complete the full workflow quickly in Colab or Domino without waiting hours.

| Setting | Value | Meaning |
|---------|-------|---------|
| `SAMPLE_ROWS` | 15,000 | Use only 15k rows from the dataset instead of all ~80k |
| `N_TRIALS` | 10 | Ask Optuna to try 10 hyperparameter combinations (not 100+) |
| `CV_FOLDS` | 2 | Use 2-fold cross-validation (faster than the standard 5) |

> 💡 **For production:** Set `SAMPLE_ROWS = None`, `N_TRIALS = 50+`, `CV_FOLDS = 5` for more reliable results.

## Step 0 — Install Libraries & Imports

In [ ]:
# Colab installs (run once)
!pip -q install mlflow optuna lightgbm xgboost joblib scikit-learn

import os
import re
import joblib
import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, accuracy_score, classification_report

import mlflow
import mlflow.sklearn
import mlflow.pyfunc

from lightgbm import LGBMRegressor, LGBMClassifier

pd.set_option("display.max_columns", 200)

## Step 1 — Set Up MLflow

### What is MLflow doing here?

MLflow needs to know *where* to save your experiment results. We call this the **tracking URI** — it is just a folder path or a server URL.

- **Local mode** (`USE_DRIVE = False`): Results are saved to a folder on the Colab/Domino machine. They disappear when the session ends.
- **Google Drive mode** (`USE_DRIVE = True`): Results are saved to your Google Drive and persist between sessions.
- **In Domino / Production**: You would replace the local path with your MLflow server URL, e.g. `mlflow.set_tracking_uri("http://your-mlflow-server:5000")`

After setting the URI, we create an **Experiment** — think of it as a project folder that groups all your related training runs together.

In [ ]:
USE_DRIVE = False  # set False if you don't care about persistence

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MLRUNS_DIR = "/content/drive/MyDrive/mlruns_irish_home_prices_light"
else:
    MLRUNS_DIR = "/content/mlruns_irish_home_prices_light"

os.makedirs(MLRUNS_DIR, exist_ok=True)
mlflow.set_tracking_uri("file://" + os.path.abspath(MLRUNS_DIR))

EXPERIMENT_NAME = "colab-irish-home-prices-light"
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLFLOW_TRACKING_URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)
print("MLruns dir:", MLRUNS_DIR)

## Step 2 — Load the Irish Property Price Register

### About the Dataset
The Irish Property Price Register is a public record of every residential property sold in Ireland. It contains the sale date, address, county, price, and a few other fields.

### What this code does
The data arrives with messy column names that contain special characters (like the € symbol), hidden whitespace, and encoding issues. The code below:
1. Downloads the CSV directly from the Irish government's open data portal
2. Cleans all column names so they are simple lowercase strings
3. Automatically detects which column is which (date, price, address, etc.) — making the code robust to minor format changes
4. Cleans the price column (removes € signs and commas, converts to a number)
5. Parses the date column into a proper datetime format
6. Drops any rows where price, date, or address is missing

In [ ]:
# --- Robust load + normalize + column mapping (copy/paste as ONE cell) ---

import re
import unicodedata
import numpy as np
import pandas as pd

DATA_URL = "https://data.smartdublin.ie/dataset/b0dd7d39-8eb5-4710-b46c-6a0db49e64af/resource/43209239-0ee7-4c8b-9599-79a54b61dd01/download/ppr-2021.csv"

# Read with Windows-1252 to correctly decode the € symbol in headers/values
df = pd.read_csv(DATA_URL, encoding="cp1252", low_memory=False)

# Strip hidden characters / whitespace
df.columns = (
    df.columns.astype(str)
      .str.replace("\ufeff", "", regex=False)   # BOM
      .str.replace("\r", "", regex=False)       # CR
      .str.replace("\n", "", regex=False)       # LF
      .str.strip()
)

# Strong normalizer: drop non-ascii (e.g., €), keep only a-z0-9 as underscores, trim underscores
def normalize_col(c: str) -> str:
    c = unicodedata.normalize("NFKD", str(c))
    c = c.encode("ascii", "ignore").decode("ascii")  # drops €, accents, etc.
    c = c.strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

df.columns = [normalize_col(c) for c in df.columns]
print("Normalized columns:", df.columns.tolist())

# Robust detection (works even if minor schema changes happen)
def first_match(prefix: str) -> str:
    matches = [c for c in df.columns if c.startswith(prefix)]
    if not matches:
        raise KeyError(f"No column starts with '{prefix}'. Available columns: {df.columns.tolist()}")
    return matches[0]

COL_DATE   = first_match("date_of_sale")
COL_PRICE  = first_match("price")
COL_ADDR   = first_match("address")
COL_COUNTY = first_match("county") if any(c.startswith("county") for c in df.columns) else None
COL_POSTAL = first_match("eircode") if any(c.startswith("eircode") for c in df.columns) else None
COL_NFMP   = first_match("not_full_market_price") if any(c.startswith("not_full_market_price") for c in df.columns) else None
COL_VAT    = first_match("vat_exclusive") if any(c.startswith("vat_exclusive") for c in df.columns) else None
COL_DESC   = first_match("description_of_property") if any(c.startswith("description_of_property") for c in df.columns) else None
COL_SIZE   = first_match("property_size_description") if any(c.startswith("property_size_description") for c in df.columns) else None

print("Resolved:")
print("  COL_DATE  =", COL_DATE)
print("  COL_PRICE =", COL_PRICE)
print("  COL_ADDR  =", COL_ADDR)

# Clean price -> float
df[COL_PRICE] = (
    df[COL_PRICE].astype(str)
      .str.replace(r"[^\d.]", "", regex=True)
      .replace("", np.nan)
)
df[COL_PRICE] = pd.to_numeric(df[COL_PRICE], errors="coerce")

# Parse date
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce", dayfirst=True)

# Drop rows missing critical fields
df = df.dropna(subset=[COL_PRICE, COL_DATE, COL_ADDR]).copy()
print("After dropping missing price/date/address:", df.shape)

df.head(3)

## Step 3 — Sampling & Feature Engineering

### Why sample?
The full dataset has ~80,000 rows. Training on all of them with hyperparameter tuning would take a long time in Colab. We randomly sample 15,000 rows to keep it fast while still demonstrating all the concepts correctly.

### What is Feature Engineering?
The raw dataset does not have many numeric columns — most of the useful information is hidden in the data. Feature engineering means **creating new columns** that make patterns easier for the model to detect.

Here we extract from the sale date:
- `sale_year` — property prices have a strong trend over time (prices in 2010 vs 2021 differ enormously)
- `sale_month` — there may be seasonal patterns (more sales in spring/summer)
- `sale_dayofweek` — a weaker signal, but included for completeness

From the address text:
- `address_len` — longer addresses often indicate more specific/urban locations
- `address_has_number` — addresses with a house number (e.g. "14 Oak Road") tend to be urban; without numbers often rural

In [ ]:
SAMPLE_ROWS = 15000  # set None to use full dataset (slower)

if SAMPLE_ROWS is not None and len(df) > SAMPLE_ROWS:
    df = df.sample(n=SAMPLE_ROWS, random_state=42).copy()

print("Using shape:", df.shape)

df["sale_year"] = df[COL_DATE].dt.year.astype("Int64")
df["sale_month"] = df[COL_DATE].dt.month.astype("Int64")
df["sale_dayofweek"] = df[COL_DATE].dt.dayofweek.astype("Int64")

df["address_len"] = df[COL_ADDR].astype(str).str.len()
df["address_has_number"] = df[COL_ADDR].astype(str).str.contains(r"\d+", regex=True).astype(int)

display(df[[COL_PRICE, COL_DATE, COL_ADDR, "sale_year", "sale_month", "address_len"]].head(3))

## Step 4 — Create Targets: Regression & Classification

### Regression Target: `y_reg`
We want to predict the actual property price in euros. This is a **regression** problem — the output is a continuous number (e.g. €285,000).

### Classification Target: `y_clf` (is_expensive)
We also want to predict whether a property is in the **top 25% of prices** — a binary classification (expensive = 1, not expensive = 0).

We use the **75th percentile** of price as the threshold. Any property priced at or above this threshold is labelled expensive.

> 💡 **Why classify instead of just predicting price?**  
> Sometimes a business needs a simple yes/no decision — "Should this loan be approved?" or "Is this claim high-value?" — rather than an exact number. Classification models are built to answer that kind of question directly.

In [ ]:
y_reg = df[COL_PRICE].astype(float)

threshold = float(y_reg.quantile(0.75))
df["is_expensive"] = (y_reg >= threshold).astype(int)
y_clf = df["is_expensive"]

print("Price threshold (top 25%):", f"€{threshold:,.0f}")
print("Class balance (0=not expensive, 1=expensive):\n", y_clf.value_counts(normalize=True).round(3))

## Step 5 — Train/Test Split & Preprocessing Setup

### Why split the data?
We hold back 20% of the data as a **test set** — data the model never sees during training. This lets us get an honest measure of how well the model performs on new, unseen data.

### What is a ColumnTransformer?
Different columns need different preprocessing treatment:
- **Numeric columns** (e.g. `sale_year`, `address_len`): We fill missing values with the median, then leave as-is. LightGBM handles numeric data natively.
- **Categorical columns** (e.g. `county`, `description_of_property`): We fill missing values with the most common value, then apply **One-Hot Encoding** — which converts text categories like "Dublin" into a set of 0/1 columns the model can read.

The `ColumnTransformer` applies the right treatment to each group of columns simultaneously.

> ⚠️ **Important:** The preprocessor is only **fitted** (learns the median values, learns the category lists) on the **training data**. It then applies those learned transformations to both training and test data. This prevents **data leakage** — accidentally letting test data influence training.

In [ ]:
base_features = ["sale_year", "sale_month", "sale_dayofweek", "address_len", "address_has_number"]
cat_candidates = [COL_COUNTY, COL_POSTAL, COL_DESC, COL_SIZE, COL_NFMP, COL_VAT]
cat_features = [c for c in cat_candidates if c is not None]

FEATURES = base_features + cat_features
X = df[FEATURES].copy()

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# Define the preprocessing steps
preprocess = ColumnTransformer(
    transformers=[
        # Numeric columns: fill missing values with the median number
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), base_features),
        # Categorical columns: fill missing with most common value, then one-hot encode
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ],
    remainder="drop"
)

print("Feature groups:")
print("  Numeric  :", base_features)
print("  Categorical:", cat_features)
print(f"\nTrain size: {len(X_train):,} | Test size: {len(X_test):,}")

## Step 6 — Hyperparameter Tuning with Optuna

### What is a Hyperparameter?
A hyperparameter is a setting you choose **before** training begins — it controls *how* the model learns, not *what* it learns.

For example, LightGBM has:
- `n_estimators` — how many trees to build (more trees = more complex model, but slower and risk of overfitting)
- `learning_rate` — how big each step is when learning (smaller = more careful but needs more trees)
- `num_leaves` — how many decision points each tree has (more = more complex patterns captured)
- `reg_alpha` / `reg_lambda` — regularisation to penalise overly complex trees (prevents overfitting)

Getting these settings right is critical. Too simple and the model misses patterns (underfitting). Too complex and it memorises the training data but fails on new data (overfitting).

### How Optuna Works
Instead of you manually trying combinations, Optuna:
1. **Defines a search space** for each hyperparameter (e.g. learning_rate between 0.01 and 0.2)
2. **Tries a combination** — trains the model, measures cross-validation score
3. **Learns from results** — uses a smart algorithm (TPE sampler) to focus on promising areas of the search space
4. **Repeats** for `N_TRIALS` iterations
5. **Returns the best combination** found

Each iteration is called a **trial**. Each `trial.suggest_*` call asks Optuna to pick a value for one hyperparameter.

### Cross-Validation (`cv=CV_FOLDS`)
Inside each trial, instead of training once, we split the training data into `CV_FOLDS` parts, train on all-but-one and test on the held-out fold, then repeat. This gives a more reliable estimate of performance than a single train/test split.

In [ ]:
N_TRIALS = 10
CV_FOLDS = 2
RANDOM_STATE = 42

def tune_lgbm_regression_fast(X_train, y_train):
    """
    Run Optuna hyperparameter search for LightGBM regression.
    Returns the best hyperparameter dictionary found.
    Objective: minimise Root Mean Squared Error (RMSE).
    Lower RMSE = predictions are closer to the true prices.
    """
    def objective(trial: optuna.Trial) -> float:
        # Optuna proposes a set of hyperparameters for this trial
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 200, 800),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 16, 128),
            "max_depth":         trial.suggest_int("max_depth", -1, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample":         trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
            "random_state": RANDOM_STATE, "n_jobs": -1,
        }
        # Build a pipeline combining preprocessing + model for fair cross-validation
        pipe = Pipeline([("prep", preprocess), ("model", LGBMRegressor(**params))])
        scores = cross_val_score(pipe, X_train, y_train, cv=CV_FOLDS, scoring="neg_root_mean_squared_error")
        return -scores.mean()  # minimise RMSE (Optuna minimises, so we negate)

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"  Best RMSE: €{study.best_value:,.0f} | Best params: {study.best_params}")
    return study.best_params


def tune_lgbm_classification_fast(X_train, y_train):
    """
    Run Optuna hyperparameter search for LightGBM classification.
    Returns the best hyperparameter dictionary found.
    Objective: maximise AUC-ROC (higher = better at ranking expensive vs cheap).
    """
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 200, 800),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 16, 128),
            "max_depth":         trial.suggest_int("max_depth", -1, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample":         trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-6, 5.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
            "random_state": RANDOM_STATE, "n_jobs": -1,
        }
        pipe = Pipeline([("prep", preprocess), ("model", LGBMClassifier(**params))])
        scores = cross_val_score(pipe, X_train, y_train, cv=CV_FOLDS, scoring="roc_auc")
        return scores.mean()  # maximise AUC

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"  Best AUC: {study.best_value:.4f} | Best params: {study.best_params}")
    return study.best_params

---
## 🅰️ Option A — sklearn Pipeline (Preprocessing Baked In)

### The Big Idea
In Option A, the **preprocessor and model are glued together** inside a scikit-learn `Pipeline` object. The pipeline has two named steps:
1. `"prep"` — the ColumnTransformer that cleans and encodes raw data
2. `"model"` — the LightGBM estimator

When you call `pipeline.fit(X_raw, y)`, it automatically fits the preprocessor first, then feeds the transformed data to the model. When you call `pipeline.predict(X_raw)`, it transforms first, then predicts.

### What gets registered in MLflow?
The **entire Pipeline object** — preprocessing + model — is saved as a single artifact. When you load it later, you get back the full pipeline. You can call `.predict()` on raw, unprocessed data directly.

```
Raw data → [Pipeline: prep → model] → Prediction
```

### When to use Option A
- Simpler projects where preprocessing is tightly coupled to the model
- When you want a single, portable artifact
- When you are using standard scikit-learn compatible transformers

### MLflow concepts used here
- `mlflow.start_run()` — opens a new "run" (a single experiment instance with its own ID)
- `mlflow.log_params()` — saves the hyperparameters Optuna found
- `mlflow.log_metric()` — saves accuracy metrics (RMSE, R2, AUC)
- `mlflow.set_tag()` — adds searchable labels to this run
- `mlflow.sklearn.log_model()` — saves the entire sklearn Pipeline object
- `mlflow.register_model()` — promotes the saved model into the **Model Registry** with a human-readable name and version number

In [ ]:
MODEL_NAME_REG_A = "colab_irish_home_reg_A"
MODEL_NAME_CLF_A = "colab_irish_home_clf_A"

# ── Option A: Regression ──────────────────────────────────────────────
print("Running Optuna for Regression (Option A)...")
best_reg = tune_lgbm_regression_fast(X_train, y_reg_train)
best_reg.update({"random_state": RANDOM_STATE, "n_jobs": -1})

# Build the full pipeline: preprocessing + LightGBM regressor
reg_pipe = Pipeline([("prep", preprocess), ("model", LGBMRegressor(**best_reg))])

with mlflow.start_run(run_name="A_reg_pipeline_light") as run:
    # Train the pipeline — prep.fit_transform() runs first, then model.fit()
    reg_pipe.fit(X_train, y_reg_train)

    # Evaluate on the held-out test set — prep.transform() runs first, then model.predict()
    preds = reg_pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    r2 = r2_score(y_reg_test, preds)

    # Log everything to MLflow
    mlflow.log_params({f"reg__{k}": v for k, v in best_reg.items()})  # hyperparams
    mlflow.log_metric("rmse", rmse)   # how wrong predictions are on average (lower = better)
    mlflow.log_metric("r2", r2)       # % of variance explained (1.0 = perfect, 0 = no better than mean)
    mlflow.set_tag("task", "regression")
    mlflow.set_tag("pattern", "optionA_pipeline")

    # Save the entire pipeline as a single artifact
    mlflow.sklearn.log_model(reg_pipe, artifact_path="model", input_example=X_train.head(3))
    run_id_reg_A = run.info.run_id

# Register in the Model Registry — gives it a name and version number
reg_A = mlflow.register_model(f"runs:/{run_id_reg_A}/model", MODEL_NAME_REG_A)
print(f"\n✅ Option A Regression registered: {reg_A.name} v{reg_A.version}")
print(f"   RMSE: €{rmse:,.0f} | R²: {r2:.3f}")

In [ ]:
# ── Option A: Classification ──────────────────────────────────────────
print("Running Optuna for Classification (Option A)...")
best_clf = tune_lgbm_classification_fast(X_train, y_clf_train)
best_clf.update({"random_state": RANDOM_STATE, "n_jobs": -1})

clf_pipe = Pipeline([("prep", preprocess), ("model", LGBMClassifier(**best_clf))])

with mlflow.start_run(run_name="A_clf_pipeline_light") as run:
    clf_pipe.fit(X_train, y_clf_train)

    # predict_proba returns probability of being in each class
    # [:, 1] = probability of being "expensive" (class 1)
    proba = clf_pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)  # convert probability to 0/1 decision

    auc = roc_auc_score(y_clf_test, proba)
    acc = accuracy_score(y_clf_test, pred)

    mlflow.log_params({f"clf__{k}": v for k, v in best_clf.items()})
    mlflow.log_metric("auc", auc)       # AUC: 1.0 = perfect ranking, 0.5 = random
    mlflow.log_metric("accuracy", acc)  # fraction of predictions that were correct
    mlflow.set_tag("task", "classification")
    mlflow.set_tag("pattern", "optionA_pipeline")

    mlflow.sklearn.log_model(clf_pipe, artifact_path="model", input_example=X_train.head(3))
    run_id_clf_A = run.info.run_id

clf_A = mlflow.register_model(f"runs:/{run_id_clf_A}/model", MODEL_NAME_CLF_A)
print(f"\n✅ Option A Classification registered: {clf_A.name} v{clf_A.version}")
print(f"   AUC: {auc:.3f} | Accuracy: {acc:.3f}")

# ── Inference Demo: Option A ──────────────────────────────────────────
print("\n--- Inference Demo (Option A) ---")
# Load the pipeline directly from the registry using its name + version
loaded_clf_A = mlflow.sklearn.load_model(f"models:/{MODEL_NAME_CLF_A}/{clf_A.version}")
# Pass RAW data — the pipeline handles preprocessing internally
sample_probas = loaded_clf_A.predict_proba(X_test.head(5))[:, 1]
print("Probability of being 'expensive' for first 5 test samples:")
for i, p in enumerate(sample_probas):
    label = "Expensive 🏠" if p >= 0.5 else "Not expensive"
    print(f"  Sample {i+1}: {p:.3f} → {label}")

---
## 🅱️ Option B — pyfunc Wrapper (Separate Preprocessor)

### The Big Idea
In Option B, the **preprocessor and model are saved as separate files**. A custom Python class (`PreprocessAndPredict`) acts as the glue — it loads both files and coordinates them at inference time.

This approach uses MLflow's **pyfunc** (Python function) flavour, which is MLflow's most flexible model type. Instead of saving a specific model type (sklearn, XGBoost etc.), you define exactly what happens when someone calls `.predict()`.

### What gets saved?
Two separate `.joblib` files:
- `preprocessor.joblib` — the fitted ColumnTransformer (knows the median values, the category lists, etc.)
- `estimator.joblib` — the fitted LightGBM model (the learned trees)

Both are registered together under one MLflow model entry, but they can be loaded independently.

```
Raw data → [pyfunc: prep.transform() → estimator.predict()] → Prediction
```

### The `pyfunc` class — 3 key methods
- `__init__()` — initialises the class (stores configuration like `return_proba`)
- `load_context(context)` — called by MLflow when the model is loaded; uses `context.artifacts` to find and load the `.joblib` files
- `predict(context, model_input)` — called when someone requests a prediction; runs preprocessing, then prediction

### When to use Option B
- When preprocessing is complex and managed separately (e.g. in a Domino pipeline or Spark job)
- When you want to **swap the preprocessor** without retraining the model (or vice versa)
- When you need full control over inference logic (e.g. returning probabilities, applying business rules)
- When using non-sklearn models that don't fit the sklearn Pipeline API

### Key difference from Option A at inference
- Option A: `mlflow.sklearn.load_model(uri)` returns a sklearn Pipeline object
- Option B: `mlflow.pyfunc.load_model(uri)` returns a generic Python model object — but `.predict()` works the same way from the outside

In [ ]:
class PreprocessAndPredict(mlflow.pyfunc.PythonModel):
    """
    Custom pyfunc model that coordinates a separate preprocessor and estimator.

    This class is the key to Option B. MLflow serialises this class definition
    alongside the artifact files. When loaded, MLflow calls load_context() to
    restore the preprocessor and estimator from disk, then predict() is called
    for each inference request.

    Args:
        return_proba: If True (classification), return probability of class=1.
                      If False (regression), return raw numeric predictions.
    """
    def __init__(self, return_proba: bool = False):
        self.return_proba = return_proba
        self.prep = None    # will be loaded from disk in load_context
        self.model = None   # will be loaded from disk in load_context

    def load_context(self, context):
        """
        Called automatically by MLflow when the model is loaded from the registry.
        context.artifacts is a dict mapping artifact names to their local file paths.
        """
        import joblib
        self.prep = joblib.load(context.artifacts["preprocessor"])  # load the ColumnTransformer
        self.model = joblib.load(context.artifacts["estimator"])    # load the LightGBM model

    def predict(self, context, model_input: pd.DataFrame):
        """
        Called when .predict() is invoked on the loaded model.
        model_input is raw, unprocessed data — same as Option A.
        """
        Xp = self.prep.transform(model_input)   # step 1: preprocess
        if self.return_proba and hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(Xp)[:, 1]  # probability of expensive
        return self.model.predict(Xp)   # step 2: predict

In [ ]:
MODEL_NAME_REG_B = "colab_irish_home_reg_B"
MODEL_NAME_CLF_B = "colab_irish_home_clf_B"

# ── Option B: Regression ──────────────────────────────────────────────
print("Running Optuna for Regression (Option B)...")
best_reg_b = tune_lgbm_regression_fast(X_train, y_reg_train)
best_reg_b.update({"random_state": RANDOM_STATE, "n_jobs": -1})

# Fit the preprocessor SEPARATELY — not inside a Pipeline
prep_fit = preprocess.fit(X_train, y_reg_train)
Xtr = prep_fit.transform(X_train)   # transform training data
Xte = prep_fit.transform(X_test)    # transform test data using SAME fitted preprocessor

# Fit the model on the already-transformed data
est = LGBMRegressor(**best_reg_b, verbosity=-1).fit(Xtr, y_reg_train)
preds = est.predict(Xte)
rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
r2 = r2_score(y_reg_test, preds)

# Save both as separate files on disk
os.makedirs("artifacts_reg", exist_ok=True)
prep_path = "artifacts_reg/preprocessor.joblib"
est_path  = "artifacts_reg/estimator.joblib"
joblib.dump(prep_fit, prep_path)   # save fitted preprocessor
joblib.dump(est, est_path)         # save fitted model

with mlflow.start_run(run_name="B_reg_pyfunc_light") as run:
    mlflow.log_params({f"reg__{k}": v for k, v in best_reg_b.items()})
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.set_tag("task", "regression")
    mlflow.set_tag("pattern", "optionB_pyfunc")

    # Register using pyfunc — provide the custom class + both artifact file paths
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=PreprocessAndPredict(return_proba=False),
        artifacts={
            "preprocessor": prep_path,   # MLflow copies these files into the run artifact store
            "estimator": est_path
        },
        input_example=X_train.head(3),
    )
    run_id_reg_B = run.info.run_id

reg_B = mlflow.register_model(f"runs:/{run_id_reg_B}/model", MODEL_NAME_REG_B)
print(f"\n✅ Option B Regression registered: {reg_B.name} v{reg_B.version}")
print(f"   RMSE: €{rmse:,.0f} | R²: {r2:.3f}")

# ── Inference Demo: Option B Regression ──────────────────────────────
print("\n--- Inference Demo (Option B Regression) ---")
# Use pyfunc.load_model — works for ANY MLflow model flavour
loaded_reg_B = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME_REG_B}/{reg_B.version}")
# Same raw input as Option A — pyfunc wrapper handles preprocessing internally
sample_preds = loaded_reg_B.predict(X_test.head(5))
print("Predicted prices for first 5 test samples:")
for i, p in enumerate(sample_preds):
    print(f"  Sample {i+1}: €{p:,.0f}")

In [ ]:
# ── Option B: Classification ──────────────────────────────────────────
print("Running Optuna for Classification (Option B)...")
best_clf_b = tune_lgbm_classification_fast(X_train, y_clf_train)
best_clf_b.update({"random_state": RANDOM_STATE, "n_jobs": -1})

prep_fit_c = preprocess.fit(X_train, y_clf_train)
Xtr = prep_fit_c.transform(X_train)
Xte = prep_fit_c.transform(X_test)

est_c = LGBMClassifier(**best_clf_b).fit(Xtr, y_clf_train)
proba = est_c.predict_proba(Xte)[:, 1]
pred  = (proba >= 0.5).astype(int)

auc = roc_auc_score(y_clf_test, proba)
acc = accuracy_score(y_clf_test, pred)

os.makedirs("artifacts_clf", exist_ok=True)
prep_path_c = "artifacts_clf/preprocessor.joblib"
est_path_c  = "artifacts_clf/estimator.joblib"
joblib.dump(prep_fit_c, prep_path_c)
joblib.dump(est_c, est_path_c)

with mlflow.start_run(run_name="B_clf_pyfunc_light_proba") as run:
    mlflow.log_params({f"clf__{k}": v for k, v in best_clf_b.items()})
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.set_tag("task", "classification")
    mlflow.set_tag("pattern", "optionB_pyfunc")

    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=PreprocessAndPredict(return_proba=True),  # return probabilities
        artifacts={"preprocessor": prep_path_c, "estimator": est_path_c},
        input_example=X_train.head(3),
    )
    run_id_clf_B = run.info.run_id

clf_B = mlflow.register_model(f"runs:/{run_id_clf_B}/model", MODEL_NAME_CLF_B)
print(f"\n✅ Option B Classification registered: {clf_B.name} v{clf_B.version}")
print(f"   AUC: {auc:.3f} | Accuracy: {acc:.3f}")

# ── Inference Demo: Option B Classification ───────────────────────────
print("\n--- Inference Demo (Option B Classification) ---")
loaded_clf_B = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME_CLF_B}/{clf_B.version}")
# Returns probabilities because return_proba=True was set in the pyfunc class
sample_probas_B = loaded_clf_B.predict(X_test.head(5))
print("Probability of being 'expensive' (Option B pyfunc):")
for i, p in enumerate(sample_probas_B):
    label = "Expensive 🏠" if p >= 0.5 else "Not expensive"
    print(f"  Sample {i+1}: {p:.3f} → {label}")

---
## 📋 Summary & Comparison

### Option A vs Option B — Side by Side

| | **Option A — sklearn Pipeline** | **Option B — pyfunc Wrapper** |
|---|---|---|
| **Preprocessing location** | Inside the sklearn Pipeline | Separate `.joblib` file |
| **Registration method** | `mlflow.sklearn.log_model()` | `mlflow.pyfunc.log_model()` |
| **Loading method** | `mlflow.sklearn.load_model()` | `mlflow.pyfunc.load_model()` |
| **Inference input** | Raw, unprocessed data ✅ | Raw, unprocessed data ✅ |
| **Flexibility** | Lower — preprocessor and model are inseparable | Higher — swap either component independently |
| **Best for** | Standard sklearn-compatible workflows | Complex preprocessing, Domino pipelines, custom inference logic |
| **Code complexity** | Simpler | Slightly more code (need to write the pyfunc class) |

### Key Takeaway
**From the outside, both options look identical at inference time.** You always pass raw data in and get predictions out. The difference is entirely internal — how the preprocessing is packaged and stored.

---

### Workflow Summary

```
1. Load raw data  →  Feature engineering  →  Train/test split
2. Define preprocessing (ColumnTransformer)
3. Run Optuna → finds best hyperparameters
4. Option A: wrap preprocessor + model in sklearn Pipeline → log + register
   Option B: fit preprocessor separately → save both as .joblib → log via pyfunc + register
5. Inference: load from MLflow registry → .predict(raw_data) → predictions
```

### MLflow Model Registry — What Happens
When you call `mlflow.register_model()`, MLflow:
- Creates a named entry in the **Model Registry** (e.g. `colab_irish_home_reg_A`)
- Assigns it **version 1** (or increments the version if the name already exists)
- Links it to the specific training **run** where the model was logged
- Stores all hyperparameters, metrics, and the model artifact together

You can then load any version by name: `mlflow.sklearn.load_model("models:/colab_irish_home_reg_A/1")`  
Or always get the latest: `mlflow.sklearn.load_model("models:/colab_irish_home_reg_A/latest")`